# DNA Sequence Analysis & Mutation Simulation
### Python portfolio sample · Daria Diablo

**Project overview.** This notebook demonstrates how Python can be used to inspect DNA sequences, simulate random mutations over successive reproduction cycles, and estimate how frequently a specified DNA subsequence is affected. It is based on programming work I completed during my Biomedical Science degree at King's College London.

**Portfolio adaptation.** I have reorganised the working logic from my university formative coursework into a self-contained example for demonstration. The example DNA sequence below is **illustrative**: the original coursework input text files were not included in this submission. The simulation uses `M` as a *mutation marker*, not as a biological nucleotide substitution.

**Key concepts:** string processing · loops and conditionals · probability simulation · reproducible results · checking a DNA subsequence

## 1. Define and validate an example DNA sequence

The subsequence used in my coursework is embedded once within clearly labelled synthetic flanking DNA. These values replace external text files so that the notebook can be run independently.

In [1]:
import random

# Illustrative input: synthetic flanks around the subsequence used in coursework.
critical_subsequence = "AATAGAAGCTCGCGCATAAAAGAGCTCCGCT"
dna_sequence = "ACGT" * 10 + critical_subsequence + "TGCA" * 11

# Check the input before running simulations.
assert set(dna_sequence).issubset(set("ACGT")), "Invalid DNA nucleotide found."
assert dna_sequence.count(critical_subsequence) == 1, "Expected one target subsequence."

print(f"DNA sequence length: {len(dna_sequence)} nucleotides")
print(f"Subsequence length: {len(critical_subsequence)} nucleotides")
print(f"Subsequence present initially: {critical_subsequence in dna_sequence}")

DNA sequence length: 115 nucleotides
Subsequence length: 31 nucleotides
Subsequence present initially: True


## 2. Model mutations over 1,000 reproduction cycles

The model checks each base once per reproduction cycle and independently gives it a **1 in 10,000** probability of being marked as mutated. Once a position is marked `M`, it remains marked in later cycles. This is a simplified *mutation detection* model rather than a model of real base substitutions.

The mutation logic below follows my coursework solution, refactored into a function so that it can be reused and tested without repeated code.

In [2]:
def simulate_mutations(sequence, cycles=1000, rng=None):
    """Mark randomly mutated positions with 'M' over successive cycles.

    Every position has a 1/10,000 chance of mutation in each cycle.
    The function returns a new sequence without changing the input.
    """
    if rng is None:
        rng = random.Random()

    mutated_sequence = sequence

    for cycle in range(cycles):
        next_sequence = []

        for nucleotide in mutated_sequence:
            # Random integer 1 corresponds to a 1-in-10,000 mutation event.
            if rng.randint(1, 10000) == 1:
                next_sequence.append("M")
            else:
                next_sequence.append(nucleotide)

        mutated_sequence = "".join(next_sequence)

    return mutated_sequence

## 3. Inspect one simulation

A fixed random seed makes this illustration reproducible. The original sequence stays unchanged; `M` indicates the positions that experienced at least one mutation event.

In [3]:
example_rng = random.Random(2026)
example_result = simulate_mutations(dna_sequence, cycles=1000, rng=example_rng)

print("Initial sequence: ", dna_sequence)
print("Final sequence:   ", example_result)
print("Mutated positions:", example_result.count("M"))
print("Target subsequence mutated:", critical_subsequence not in example_result)

Initial sequence:  ACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTAATAGAAGCTCGCGCATAAAAGAGCTCCGCTTGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCA
Final sequence:    AMGTAMGTACGTACGTACGTACGMACGTACGTAMGTACGTAATAGAAGCTCGCGCATAAAAGAMMTCCGCTMGCATGCATGCATGCATGCATGMATGCATGCATGCATGCATGCA
Mutated positions: 8
Target subsequence mutated: True


## 4. Estimate the probability of subsequence mutation

The simulation is repeated **100 times**, starting with the same original DNA sequence on each run. We record whether the exact target subsequence remains in the final DNA sequence, then calculate the proportion of runs in which it is no longer present.

This is a **Monte Carlo estimate** based on 100 trials, so the result may differ between random seeds or larger experiments. The simulation does not estimate mutation effects on reproduction or organism survival.

In [4]:
trials = 100
cycles_per_trial = 1000
rng = random.Random(42)
mutation_count = 0

for simulation in range(trials):
    # Restart with the original unmodified DNA for every independent trial.
    final_sequence = simulate_mutations(dna_sequence, cycles_per_trial, rng=rng)

    if critical_subsequence not in final_sequence:
        mutation_count += 1

estimated_probability = mutation_count / trials

print(f"Simulations run: {trials}")
print(f"Simulations with target mutation: {mutation_count}")
print(f"Estimated probability of target mutation: {estimated_probability:.1%}")

Simulations run: 100
Simulations with target mutation: 94
Estimated probability of target mutation: 94.0%


## 5. Interpretation and limitations

- The simulation estimates how often at least one position in a specific DNA subsequence is marked as mutated after 1,000 cycles.
- The result depends on the subsequence length, number of cycles, per-base mutation probability and random sampling. A different seed or number of trials may produce a slightly different estimate.
- To keep the model simple, the mutation marker `M` is never converted back to `A`, `C`, `G` or `T`. In real biology, nucleotide substitutions can have different consequences, and mutation probabilities may vary by position and context.
- **Possible next step:** increase the number of simulations, compare different mutation rates, or plot the estimated mutation probability against reproduction-cycle count.

**Skills illustrated:** Python fundamentals, iterative simulations, working with biological sequences, basic probability estimation, data validation and reproducible computational workflows.